
 **Gold Layer - Business Ready Tables**

 - Enriched sales fact table (sales + product + customer)
 - Monthly revenue summary
 - Product performance summary
 - Customer spending summary



**Enriched sales fact table**: join sales with product and customer info

In [0]:
%sql

CREATE OR REPLACE TABLE retail_store.gold.fact_sales AS
SELECT
    s.transaction_id,
    s.transaction_date,
    s.customer_id,
    c.gender,
    c.location,
    c.age,
    s.product_id,
    p.product_name,
    p.category,
    s.quantity_purchased,
    s.price,
    s.revenue,
    year(s.transaction_date) AS transaction_year,
    month(s.transaction_date) AS transaction_month,
    current_timestamp() AS _processed_timestamp
FROM retail_store.silver.sales_transactions s
LEFT JOIN retail_store.silver.product_inventory p ON s.product_id = p.product_id
LEFT JOIN retail_store.silver.customer_profiles c ON s.customer_id = c.customer_id;

num_affected_rows,num_inserted_rows


check table

In [0]:
%sql

SELECT * FROM retail_store.gold.fact_sales LIMIT 5;

transaction_id,transaction_date,customer_id,gender,location,age,product_id,product_name,category,quantity_purchased,price,revenue,transaction_year,transaction_month,_processed_timestamp
1,2023-01-01,103,Female,South,21,120,Product_120,electronics,3,30.43,91.29,2023,1,2026-03-16T11:33:14.663Z
2,2023-01-01,436,Male,West,55,126,Product_126,electronics,1,15.19,15.19,2023,1,2026-03-16T11:33:14.663Z
3,2023-01-01,861,Male,South,59,55,Product_55,electronics,3,67.76,203.28,2023,1,2026-03-16T11:33:14.663Z
4,2023-01-01,271,Female,West,50,27,Product_27,electronics,2,65.77,131.54,2023,1,2026-03-16T11:33:14.663Z
5,2023-01-01,107,Male,South,60,118,Product_118,beauty & health,1,14.55,14.55,2023,1,2026-03-16T11:33:14.663Z


**Monthly Revenue**

In [0]:
%sql
CREATE OR REPLACE TABLE retail_store.gold.monthly_revenue AS
SELECT
    transaction_year,
    transaction_month,
    COUNT(transaction_id) AS total_transactions,
    SUM(quantity_purchased) AS total_units_sold,
    ROUND(SUM(revenue), 2) AS total_revenue,
    ROUND(AVG(revenue), 2) AS avg_revenue_per_transaction
FROM retail_store.gold.fact_sales
GROUP BY transaction_year, transaction_month
ORDER BY transaction_year, transaction_month;

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT * FROM retail_store.gold.monthly_revenue;

transaction_year,transaction_month,total_transactions,total_units_sold,total_revenue,avg_revenue_per_transaction
2023,1,744,1797,104289.18,140.17
2023,2,672,1737,96690.99,143.89
2023,3,744,1812,103271.49,138.81
2023,4,720,1790,101561.09,141.06
2023,5,744,1796,102998.84,138.44
2023,6,720,1784,102210.28,141.96
2023,7,656,1609,90981.75,138.69


which products sell the most: **Product Performance**

In [0]:
%sql
CREATE OR REPLACE TABLE retail_store.gold.product_performance AS
SELECT
    product_id,
    product_name,
    category,
    COUNT(transaction_id) AS times_sold,
    SUM(quantity_purchased) AS total_units_sold,
    ROUND(SUM(revenue), 2) AS total_revenue,
    ROUND(AVG(price), 2) AS avg_price
FROM retail_store.gold.fact_sales
GROUP BY product_id, product_name, category
ORDER BY total_revenue DESC;

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT * FROM retail_store.gold.product_performance LIMIT 10;

product_id,product_name,category,times_sold,total_units_sold,total_revenue,avg_price
17,Product_17,beauty & health,39,100,9450.0,94.5
87,Product_87,clothing,35,92,7817.24,84.97
179,Product_179,clothing,31,86,7388.26,85.91
96,Product_96,home & kitchen,28,72,7132.32,99.06
54,Product_54,electronics,32,86,7052.86,82.01
187,Product_187,beauty & health,34,82,6915.88,84.34
156,Product_156,clothing,33,76,6827.84,89.84
57,Product_57,home & kitchen,33,78,6622.2,84.9
200,Product_200,clothing,24,69,6479.79,93.91
127,Product_127,electronics,29,68,6415.8,94.35


**Customer Spending**

In [0]:
%sql
CREATE OR REPLACE TABLE retail_store.gold.customer_spending AS
SELECT
    customer_id,
    location,
    gender,
    age,
    COUNT(transaction_id) AS total_orders,
    SUM(quantity_purchased) AS total_items_bought,
    ROUND(SUM(revenue), 2) AS total_spent,
    ROUND(AVG(revenue), 2) AS avg_order_value,
    MIN(transaction_date) AS first_purchase,
    MAX(transaction_date) AS last_purchase
FROM retail_store.gold.fact_sales
GROUP BY customer_id, location, gender, age
ORDER BY total_spent DESC;

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT * FROM retail_store.gold.customer_spending LIMIT 10;

customer_id,location,gender,age,total_orders,total_items_bought,total_spent,avg_order_value,first_purchase,last_purchase
936,East,Male,65,12,35,2834.47,236.21,2023-01-15,2023-07-27
664,South,Other,33,14,39,2519.04,179.93,2023-01-01,2023-07-24
670,East,Other,45,12,35,2432.15,202.68,2023-01-19,2023-07-08
39,West,Male,20,12,34,2221.29,185.11,2023-01-22,2023-05-31
435,North,Male,61,10,28,2158.98,215.9,2023-01-27,2023-07-19
958,East,Other,47,12,34,2104.71,175.39,2023-01-03,2023-07-21
364,East,Other,34,8,24,1885.4,235.68,2023-03-16,2023-07-12
407,West,Other,63,8,23,1882.26,235.28,2023-01-07,2023-07-22
921,West,Female,69,10,28,1866.29,186.63,2023-01-23,2023-07-12
75,East,Other,30,11,24,1862.73,169.34,2023-01-31,2023-07-26
